# RQ2: Bias removal (rewriting) analysis

Requires the outputs of `bash_scripts/rq2_judge.sh` (`outputs/bias_judge/gemma-4-31b/`).

1. Overall judge scores per model
2. Trigger removal and factual preservation per bias type, single vs. pairwise
3. A sample of 100 rewriting errors for qualitative inspection

In [ ]:
import os

import pandas as pd

JUDGE_DIR = "../outputs/bias_judge/gemma-4-31b"
ANALYSIS_DIR = "../outputs/analysis"
os.makedirs(ANALYSIS_DIR, exist_ok=True)

MODELS = ["gpt-5.4-mini", "claude-haiku-4.5", "gemini-3-flash", "llama-3.1-8b", "llama-3.1-70b", "llama-3.3-70b",
          "qwen3-8b", "qwen3.5-9b", "qwen3-32b", "gemma-4-12b", "gemma-4-31b"]
BIAS_TYPES = ["anchoring", "authority", "framing", "herding", "recency", "overconfidence"]

## Overall judge scores

In [ ]:
summary_df = pd.read_csv(os.path.join(JUDGE_DIR, "all_models_summary.csv"))
summary_df.set_index("judged_model").reindex(MODELS)

## Trigger removal and factual preservation per bias type

In [ ]:
trigger_results = []
facts_results = []

for model in MODELS:
    judge_file = os.path.join(JUDGE_DIR, model, "judge_results.csv")
    
    if not os.path.exists(judge_file):
        for bias in BIAS_TYPES:
            trigger_results.append({"bias_type": bias, "subset": "Single", "model": model, "value": "TBD"})
            trigger_results.append({"bias_type": bias, "subset": "Pairwise", "model": model, "value": "TBD"})
            facts_results.append({"bias_type": bias, "subset": "Single", "model": model, "value": "TBD"})
            facts_results.append({"bias_type": bias, "subset": "Pairwise", "model": model, "value": "TBD"})
        continue
    
    df = pd.read_csv(judge_file)
    
    # Determine single vs pairwise based on biases column
    df["is_pairwise"] = df["biases"].str.contains(",", na=False)
    
    for bias in BIAS_TYPES:
        # Single: biases == bias (exact match, no comma)
        df_single = df[(df["biases"] == bias)]
        # Pairwise: biases contains bias and has a comma
        df_pairwise = df[(df["biases"].str.contains(bias, na=False)) & (df["is_pairwise"])]
        
        # Trigger removal
        if len(df_single) > 0:
            trigger_single = round(df_single["trigger_removed"].mean() * 100, 1)
        else:
            trigger_single = "TBD"
        
        if len(df_pairwise) > 0:
            trigger_pairwise = round(df_pairwise["trigger_removed"].mean() * 100, 1)
        else:
            trigger_pairwise = "TBD"
        
        trigger_results.append({"bias_type": bias, "subset": "Single", "model": model, "value": trigger_single})
        trigger_results.append({"bias_type": bias, "subset": "Pairwise", "model": model, "value": trigger_pairwise})
        
        # Facts preserved
        if len(df_single) > 0:
            facts_single = round(df_single["facts_preserved"].mean() * 100, 1)
        else:
            facts_single = "TBD"
        
        if len(df_pairwise) > 0:
            facts_pairwise = round(df_pairwise["facts_preserved"].mean() * 100, 1)
        else:
            facts_pairwise = "TBD"
        
        facts_results.append({"bias_type": bias, "subset": "Single", "model": model, "value": facts_single})
        facts_results.append({"bias_type": bias, "subset": "Pairwise", "model": model, "value": facts_pairwise})

# Create pivot tables
trigger_df = pd.DataFrame(trigger_results)
trigger_pivot = trigger_df.pivot(index=["bias_type", "subset"], columns="model", values="value")
trigger_pivot = trigger_pivot[MODELS]

facts_df = pd.DataFrame(facts_results)
facts_pivot = facts_df.pivot(index=["bias_type", "subset"], columns="model", values="value")
facts_pivot = facts_pivot[MODELS]

trigger_path = os.path.join(ANALYSIS_DIR, "rewriting_trigger_removal.csv")
facts_path = os.path.join(ANALYSIS_DIR, "rewriting_facts_preserved.csv")

trigger_pivot.to_csv(trigger_path)
facts_pivot.to_csv(facts_path)

print(f"Saved to: {trigger_path}")
print(f"Saved to: {facts_path}")

print("\n=== Trigger Removal (%) ===")
print(trigger_pivot)

print("\n=== Factual Preservation (%) ===")
print(facts_pivot)

## Sample of rewriting errors

In [ ]:
all_errors = []

for model in MODELS:
    judge_file = os.path.join(JUDGE_DIR, model, "judge_results.csv")
    
    if not os.path.exists(judge_file):
        continue
    
    df = pd.read_csv(judge_file)
    
    # Trigger retained errors
    trigger_errors = df[df["trigger_removed"] == 0].copy()
    trigger_errors["error_type"] = "trigger_retained"
    trigger_errors["model"] = model
    
    # Fact damaged errors
    fact_errors = df[df["facts_preserved"] == 0].copy()
    fact_errors["error_type"] = "fact_damaged"
    fact_errors["model"] = model
    
    # Unnatural errors
    natural_errors = df[df["natural_language"] == 0].copy()
    natural_errors["error_type"] = "unnatural"
    natural_errors["model"] = model
    
    all_errors.append(trigger_errors)
    all_errors.append(fact_errors)
    all_errors.append(natural_errors)

combined_errors = pd.concat(all_errors, ignore_index=True)

# Sample 100 errors (or all if less than 100)
n_samples = min(100, len(combined_errors))
sampled_errors = combined_errors.sample(n=n_samples, random_state=42)

# Select relevant columns
output_cols = ["biases", "error_type", "original_sentence", "biased_sentence", "detected_bias_rewrite", "model"]
sampled_errors = sampled_errors[output_cols]
sampled_errors.columns = ["bias_type", "error_type", "original_sentence", "biased_sentence", "model_rewrite", "model"]

# Sort by error type and bias type for easier browsing
sampled_errors = sampled_errors.sort_values(["error_type", "bias_type"]).reset_index(drop=True)

output_path = os.path.join(ANALYSIS_DIR, "rewriting_errors_sample_100.csv")
sampled_errors.to_csv(output_path, index=False)

print(f"Saved {len(sampled_errors)} errors to: {output_path}")
print(f"\nError type distribution:")
print(sampled_errors["error_type"].value_counts())
print(f"\nBias type distribution:")
print(sampled_errors["bias_type"].value_counts())